In [1]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"chauayush","key":"f0e72f3ade58b81742f66937f4b49578"}'}

In [2]:
import os
os.listdir()

['.config', 'kaggle (1).json', 'sample_data']

In [3]:
import os, shutil

os.makedirs('/root/.kaggle', exist_ok=True)

shutil.move('kaggle (1).json', '/root/.kaggle/kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d karakaggle/kaggle-cat-vs-dog-dataset
!unzip -q kaggle-cat-vs-dog-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset
License(s): unknown
100% 787M/787M [00:52<00:00, 15.6MB/s]



In [12]:
import os
from PIL import Image

path = '/content/kagglecatsanddogs_3367a/PetImages'

for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(path, folder)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        try:
            img = Image.open(file_path)
            img.verify()  # check corruption
        except:
            os.remove(file_path)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [13]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten


In [14]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='/content/kagglecatsanddogs_3367a/PetImages',
    validation_split=0.2,
    subset='training',
    labels='inferred',
    label_mode='int',
    seed=123,
    image_size=(256, 256),
    batch_size=32
)

val_ds = keras.utils.image_dataset_from_directory(
    directory='/content/kagglecatsanddogs_3367a/PetImages',
    validation_split=0.2,
    labels='inferred',
    subset='validation',
    seed=123,
    label_mode='int',
    image_size=(256, 256),
    batch_size=32
)

Found 24959 files belonging to 2 classes.
Using 19968 files for training.
Found 24959 files belonging to 2 classes.
Using 4991 files for validation.


In [20]:
import tensorflow as tf

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [21]:
def process(image,label):
  image=tf.cast(image/255,tf.float32)
  return image,label
train_ds = train_ds.map(process)
val_ds = val_ds.map(process)


In [22]:
model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(54,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [23]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 54)             │         6,966 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            55 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,845,997 (56.63 MB)

 Trainable params: 14,845,997 (56.63 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [25]:
history=model.fit(train_ds,epochs=10,validation_data=val_ds)

Epoch 1/10
    618/Unknown 45s 66ms/step - accuracy: 0.5037 - loss: 0.6933

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


618/618 ━━━━━━━━━━━━━━━━━━━━ 56s 85ms/step - accuracy: 0.5048 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6937
Epoch 2/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 49s 80ms/step - accuracy: 0.5012 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6935
Epoch 3/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 81s 131ms/step - accuracy: 0.5021 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6935
Epoch 4/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 50s 80ms/step - accuracy: 0.5007 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6936
Epoch 5/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 49s 79ms/step - accuracy: 0.5035 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6936
Epoch 6/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 49s 80ms/step - accuracy: 0.5036 - loss: 0.6932 - val_accuracy: 0.4891 - val_loss: 0.6936
Epoch 7/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 86s 139ms/step - accuracy: 0.5040 - loss: 0.6931 - val_accuracy: 0.4891 - val_loss: 0.6936
Epoch 8/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 48s 77ms/step - accuracy: 0.5033 - loss: 0.6932 - val_accur